# GIAI ĐOẠN 4: Chạy Thực Nghiệm Chính (Đã tối ưu)

## Experiment Design
- **2 Pipelines**: Baseline (uncertainty-only) vs Stratified (metadata + uncertainty)
- **2 Rounds**: Active Learning rounds (0, 1, 2)
- **5 Epochs**: Training epochs per round
- **3 Seeds**: 42, 123, 456
- **6 Experiments** total

## Tối ưu đã áp dụng (vòng trước mất 40h, giờ ~2.5h/experiment)
| Thay đổi | Trước | Sau |
|-----------|-------|-----|
| Batch size predict | 32 | 256 |
| Memory | giữ toàn bộ Results | `stream=True` |
| Precision | FP32 | FP16 (`half=True`) |
| GC | mỗi batch (1917 lần) | mỗi ~10 chunk |
| Check file tồn tại | 61k lần `Path.exists()` | bỏ (đã lọc sẵn) |
| Parse JSON + walk ảnh | 3.5 phút × 6 | cache pickle, chỉ 1 lần |

In [ ]:
# CANH BIEN BO NO TRUOC khi import (phai chay TRUOC moi cell khac)
# glibc malloc giu heap cache -> RSS phen giong, cgroup deng lo va OOM.
# MALLOC_TRIM_THRESHOLD_=65536: ep glibc tra RAM vao OS sau mỗi 64KB free.
import os as _os
_os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
_os.environ["OMP_NUM_THREADS"] = "1"
_os.environ["MKL_NUM_THREADS"] = "1"
print("[ENV] MALLOC_TRIM_THRESHOLD_=65536, OMP/MKL threads=1")


In [ ]:
# Pin ultralytics version DĐ chứng minh chạy sống (log 21/10).
# Version mới (>=8.4.191) thay đổi dataloader -> OOM, Kernel died.
!pip install ultralytics==8.4.184 torch torchvision scikit-learn scipy pandas pyyaml psutil -q
print("Dependencies installed (ultralytics pinned to 8.4.184)")
print("Dependencies installed!")

In [ ]:
# Clone repo (xoá bản cũ để luôn lấy code mới nhất)
!rm -rf /kaggle/working/metadata-stratified-active-learning
!git clone -b develop https://github.com/nguyendinhtanloc-Org/metadata-stratified-active-learning.git
import sys
sys.path.insert(0, "/kaggle/working/metadata-stratified-active-learning/src")
sys.path.insert(0, "/kaggle/working/metadata-stratified-active-learning")
print("Repo ready")

In [ ]:
# Import modules
from active_learning.loop import ActiveLearningLoop
from sampling.baseline import baseline_sampling
from sampling.stratified import stratified_sampling
import pandas as pd
import os
import time
from datetime import datetime

print("Modules imported successfully!")

In [ ]:
# Dataset paths
METADATA_PATH = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"

IMAGE_DIRS = [
    "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainA",
    "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainB",
]

results_dir = "results/main_experiment"
os.makedirs(results_dir, exist_ok=True)
print("Paths configured")

In [ ]:
# Do RAM/CPU + GIỚI HẠN CGROUP (quan trong chuan doan OOM)
import os, psutil, subprocess
vm = psutil.virtual_memory()
print(f"[MEM] RAM host: {vm.total/1e9:.1f} GB, available: {vm.available/1e9:.1f} GB")
# psutil doc RAM HOST - khong phai gioi han cgroup cua notebook Kaggle.
# OOM Killer dung cgroup limit, nen phai doc truc tiep:
for p in ['/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes']:
    try:
        raw = open(p).read().strip()
        try:
            limit = int(raw)
            print(f"[CGROUP] memory limit: {limit/1e9:.1f} GB" if limit < 10**12 else f"[CGROUP] memory limit: unlimited ({raw})")
        except ValueError:
            print(f"[CGROUP] {p}: {raw}")
        break
    except FileNotFoundError:
        continue
print(f"[CPU] {os.cpu_count()} cores available")


In [ ]:
## Config - tối ưu cho Kaggle 12h
EXPERIMENT_CONFIG = {
    "seed": 42,
    "initial_budget": 500,
    "al_batch_size": 500,
    "train_batch_size": 16,
    "num_rounds": 2,
    "device": 0,
    "num_classes": 10,
    "stratify_fields": ["weather", "scene", "timeofday"],
    "model_name": "yolov8n",
    # --- Inference tuning ---
    "predict_batch_size": 128,         # giam 256->128: in-flight RAM ~nua, van nhanh (I/O-bound)
    "predict_workers": 0,          # workers=0: tranh CoW leak; RAM co gioi han nhan hoa   # tăng lên 512 nếu VRAM cho phép
    "predict_imgsz": 640,        # giống training -> entropy nhất quán
    # --- Local image cache (TĂNG TỐC RẤT NHIỀU) ---
    # Đọc ảnh từ /kaggle/input rất chậm (network-backed).
    # Copy 1 lần vào /kaggle/working (local SSD) -> predict nhanh hơn nhiều lần.
    # Đo thực tế: GPU inference 2ms/ảnh, tổng 160ms/ảnh -> 98% là I/O.
    "local_image_cache": None,  # TAT: /kaggle/working la RAM disk (tmpfs) -> copy anh vao do gay OOM
    # --- Fixed held-out eval set ---
    # 2000 ảnh bị loại vĨNH VIỄN khỏi pool AL -> mAP comparable mọi round/seed
    "eval_size": 2000,
    "eval_split_seed": 2024,     # cố định cho MỌI pipeline & seed
    # --- Early stopping ---
    "early_stop_patience": 0,   # 0 = tắt
    # --- Config phiên chạy mới (hybrid + candidate pool) ---
    "candidate_pool_size": 15000, # mỗi round chỉ tính uncertainty trên 15k ảnh con (seed=seed*1000+round, tái lập được)
    "hybrid_top_k_ratio": 5,      # stratified: lọc Top-(500*5)=2500 ảnh KHÓ NHẤT trước, rồi mới phân tầng bên trong
    "epochs": 15,                 # tăng từ 5 -> 15 (tập dữ liệu đa dạng + hybrid cần nhiều bước lặp hơn)
}

print("Experiment Configuration:")
for k, v in EXPERIMENT_CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
SEEDS = [42, 123, 456]
PIPELINES = ["baseline", "stratified"]
all_results = []
timing_log = []

# Nếu muốn test nhanh, đổi thành: SEEDS = [42]; PIPELINES = ["baseline"]
print(f"Total experiments: {len(PIPELINES) * len(SEEDS)}")

## RUN ALL EXPERIMENTS

Mỗi experiment lưu CSV riêng + checkpoint. Nếu session timeout, chạy lại notebook này: cell sau sẽ tự load kết quả đã có.

In [ ]:
# Khoi phuc ket qua tu Kaggle Output (cho session 2)
# /kaggle/working bi xoa sach khi session ket thuc -> ket qua session 1
# phai luu ra /kaggle/working/output/al_results roi upload lam Dataset moi.
#
# CELL NAY PHAI CHAY TRUOC VONG CHAY CHINH (cell tiep theo).
import os, glob, shutil

# Round cuoi can restore model weights (dung de val theo domain o cell sau)
LAST_ROUND = EXPERIMENT_CONFIG.get("num_rounds", 2)

restore_dirs = []
for pat in ["/kaggle/working/*/al_results",
            "/kaggle/working/output/al_results",
            "/kaggle/input/*/al_results",
            "/kaggle/input/*/*/al_results"]:
    restore_dirs.extend(glob.glob(pat))
restore_dirs = sorted(set(restore_dirs))

restored_csv = 0
restored_ckpt = 0
for d in restore_dirs:
    for f in glob.glob(os.path.join(d, "*.csv")):
        dst = os.path.join(results_dir, os.path.basename(f))
        os.makedirs(results_dir, exist_ok=True)
        shutil.copy(f, dst)
        restored_csv += 1
    # checkpoint: <exp>_checkpoint.pkl -> results/main_experiment/<exp>/checkpoint.pkl
    for f in glob.glob(os.path.join(d, "*_checkpoint.pkl")):
        exp = os.path.basename(f).replace("_checkpoint.pkl", "")
        exp_dir = os.path.join(results_dir, exp)
        os.makedirs(exp_dir, exist_ok=True)
        shutil.copy(f, os.path.join(exp_dir, "checkpoint.pkl"))
        restored_ckpt += 1

# Khoi phuc model weights (best.pt) -> dung duong dan ultralytics da dung
restored_models = 0
for d in restore_dirs:
    for f in glob.glob(os.path.join(d, "models", "*", "best.pt")):
        exp = os.path.basename(os.path.dirname(f))
        dst = os.path.join("runs/detect/experiments/runs", exp, f"round_{LAST_ROUND}", "weights")
        os.makedirs(dst, exist_ok=True)
        shutil.copy(f, os.path.join(dst, "best.pt"))
        restored_models += 1

# Khoi phuc eval set (2000 anh) -> cell domain-eval can val theo domain
restored_eval = 0
for d in restore_dirs:
    src_eval = os.path.join(d, "eval_set")
    if os.path.isdir(src_eval):
        for sub in ("images", "labels"):
            s_sub = os.path.join(src_eval, sub)
            if os.path.isdir(s_sub):
                shutil.copytree(s_sub, os.path.join("data/processed/_shared_eval", sub), dirs_exist_ok=True)
                restored_eval += 1
        restored_eval = len(glob.glob("data/processed/_shared_eval/images/*"))

print(f"[RESTORE] {restored_csv} CSV + {restored_ckpt} checkpoint + "
      f"{restored_models} model + {restored_eval} eval anh tu {len(restore_dirs)} dirs")
if restored_csv or restored_ckpt:
    for f in sorted(os.listdir(results_dir)):
        print(f"   - {f}")
else:
    print("   (khong co ket qua cu - se chay moi tu dau)")


In [ ]:
## [TAT] Copy anh vao local disk - KHONG DUOC SU DUNG
# /kaggle/working la RAM disk (tmpfs), KHONG PHAI SSD.
# Cache 61k anh -> an ~4GB RAM -> dataloader predict -> OOM -> Kernel died.
# Doc anh tu /kaggle/input cham (network) nhung AN TOAN ve RAM.
ENABLE_LOCAL_IMAGE_CACHE = False

if not ENABLE_LOCAL_IMAGE_CACHE:
    print("[CACHE] Biep qua - chi chay neu ban chinh sach thuc su co dia local (khong phai /kaggle/working)")
else:
    import os, shutil, time
    LOCAL_IMG_DIR = "data/local_images"
    if os.path.isdir(LOCAL_IMG_DIR):
        print(f"[CACHE] {LOCAL_IMG_DIR} da co {len(os.listdir(LOCAL_IMG_DIR))} anh")
    else:
        t0 = time.time()
        os.makedirs(LOCAL_IMG_DIR, exist_ok=True)
        n = 0
        for d in IMAGE_DIRS:
            if not os.path.exists(d): continue
            for f in os.listdir(d):
                if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
                dst = os.path.join(LOCAL_IMG_DIR, f)
                if os.path.exists(dst): continue
                try:
                    shutil.copy(os.path.join(d, f), dst); n += 1
                except Exception: pass
        print(f"[CACHE] {n} anh trong {time.time()-t0:.0f}s")


In [ ]:
import warnings
warnings.filterwarnings('ignore')

for pipeline in PIPELINES:
    for seed in SEEDS:
        experiment_name = f"{pipeline}_seed{seed}"
        csv_path = f"{results_dir}/{experiment_name}.csv"
        
        # Skip nếu đã có kết quả (resume)
        if os.path.exists(csv_path):
            print(f"\n[SKIP] {experiment_name} - đã có kết quả")
            try:
                all_results.append(pd.read_csv(csv_path))
            except Exception as e:
                print(f"  [WARN] Không đọc được {csv_path}: {e}")
            continue
        
        print("\n" + "="*70)
        print(f"EXPERIMENT: {experiment_name}")
        started_at = time.time()
        print("="*70)
        
        config = EXPERIMENT_CONFIG.copy()
        config["seed"] = seed
        config["use_baseline"] = (pipeline == "baseline")
        
        print(f"[INFO] Using {'BASELINE' if pipeline == 'baseline' else 'STRATIFIED'} sampling")
        
        al_loop = ActiveLearningLoop(
            config=config,
            experiment_name=experiment_name
        )
        
        try:
            results_df = al_loop.run(
                metadata_path=METADATA_PATH,
                image_dirs=IMAGE_DIRS,
                output_dir=results_dir      # -> results/main_experiment/<exp>/
            )
            
            results_df["pipeline"] = pipeline
            results_df["seed"] = seed
            results_df.to_csv(csv_path, index=False)
            print(f"\n[Saved] {csv_path}")
            all_results.append(results_df)
            
            elapsed = time.time() - started_at
            timing_log.append({
                "experiment": experiment_name,
                "elapsed_sec": round(elapsed, 1),
                "elapsed_hr": round(elapsed / 3600, 2),
            })
            print(f"\n[DONE] {experiment_name} in {elapsed/3600:.2f}h")
            
        except Exception as e:
            print(f"\n[ERROR] {experiment_name} failed: {e}")
            import traceback
            traceback.print_exc()
            continue

In [ ]:
# Timing summary
if timing_log:
    timing_df = pd.DataFrame(timing_log)
    print(timing_df.to_string(index=False))
    total_hr = timing_df['elapsed_hr'].sum()
    print(f"\nTotal: {total_hr:.2f}h")
    print(f"Avg per experiment: {timing_df['elapsed_hr'].mean():.2f}h")

## SAVE COMBINED RESULTS

In [ ]:
# Combine all results (nạp lại từ CSV để chắc chắn đủ 6 file)
combined_frames = []
for pipeline in PIPELINES:
    for seed in SEEDS:
        p = f"{results_dir}/{pipeline}_seed{seed}.csv"
        if os.path.exists(p):
            combined_frames.append(pd.read_csv(p))
            print(f"  Loaded {p}")

if combined_frames:
    combined_df = pd.concat(combined_frames, ignore_index=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    combined_path = f"{results_dir}/combined_results_{timestamp}.csv"
    combined_df.to_csv(combined_path, index=False)
    print(f"\nCombined results saved to: {combined_path}")
    
    print("\n" + "="*70)
    print("EXPERIMENT SUMMARY")
    print("="*70)
    print(f"Total experiments: {combined_df['experiment'].nunique() if 'experiment' in combined_df.columns else len(combined_frames)}")
    
    print(combined_df.groupby(['pipeline', 'round'])['map50'].agg(['mean', 'std']).round(4))
else:
    print("No results to combine!")

In [ ]:
# Luu ket qua ra Kaggle Output (de KHONG mat khi session ket thuc)
# /kaggle/working bi xoa sach khi session chet -> phai copy sang /kaggle/working/output
# Ket qua + checkpoint se xuat hien trong tab 'Output' cua notebook tren Kaggle.
import shutil, glob, os, zipfile

output_dir = "/kaggle/working/output/al_results"
os.makedirs(output_dir, exist_ok=True)

copied = 0
# 1) CSV tong hop (results/main_experiment/*.csv) - dau chinh de lay lai
for src in glob.glob("results/main_experiment/*.csv"):
    shutil.copy(src, output_dir)
    copied += 1

# 2) Checkpoint (results/main_experiment/<exp>/checkpoint.pkl) -> de resume
for src in glob.glob("results/main_experiment/*/checkpoint.pkl"):
    exp = os.path.basename(os.path.dirname(src))
    shutil.copy(src, os.path.join(output_dir, f"{exp}_checkpoint.pkl"))
    copied += 1

# 3) Model weights (best.pt round CUOI) - de cell domain-eval chay duoc o session sau
#    /kaggle/working bi xoa khi session ket thuc -> weights cung bi xoa theo.
#    best.pt ~6.5MB x 6 experiments = ~40MB, gia re so voi 5GB output limit.
LAST_ROUND = EXPERIMENT_CONFIG.get("num_rounds", 2)
for src in glob.glob(f"runs/detect/experiments/runs/*/round_{LAST_ROUND}/weights/best.pt"):
    exp = src.split("/runs/")[1].split("/")[0]
    dst_dir = os.path.join(output_dir, "models", exp)
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy(src, os.path.join(dst_dir, "best.pt"))
    copied += 1

# 4) Eval set (2000 anh + label) - cell domain-eval can images de val theo domain
if os.path.isdir("data/processed/_shared_eval"):
    eval_dst = os.path.join(output_dir, "eval_set")
    for sub in ("images", "labels"):
        if os.path.isdir(f"data/processed/_shared_eval/{sub}"):
            shutil.copytree(f"data/processed/_shared_eval/{sub}",
                            os.path.join(eval_dst, sub), dirs_exist_ok=True)
            copied += 1
    print(f"[SAVED] eval set -> {eval_dst}")

# 5) Zip de tai xuong 1 lan (Kaggle Output khong tao file .pkl rieng)
zip_path = "/kaggle/working/output/al_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _dirs, files in os.walk(output_dir):
        for f in sorted(files):
            full = os.path.join(root, f)
            zf.write(full, os.path.relpath(full, output_dir))

print(f"[SAVED] {copied} files -> {output_dir}")
for f in sorted(os.listdir(output_dir)):
    print(f"   - {f}")
print(f"\n[ZIP] {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)")
print("\n>>> Download 'al_results.zip' tu tab Output cua notebook tren Kaggle")
print(">>> Upload lai file do lam Dataset moi (ten: al_results) cho session 2")


## STATISTICAL ANALYSIS (H1 & H2)

In [ ]:
# ============================================================
# STATISTICAL ANALYSIS H1 & H2 (ban doc that CSV - khong phu thuoc
# bien trong memory, chay duoc o BATS KY session nao).
# Ket qua da luu tai results_dir (tu cell-restore neu co dataset).
# ============================================================
from scipy import stats
import numpy as np
import ast, glob, os

csv_files = sorted(glob.glob(f"{results_dir}/*.csv"))
csv_files = [f for f in csv_files
              if not os.path.basename(f).startswith(("combined", "_shared"))]
# ^ _shared_eval_metadata.csv (2000 dong metadata eval) KHONG phai ket qua
#   experiment -> loai ra, neu khong se lam H2 ra NaN.

if not csv_files:
    print("[STAT] Khong tim thay CSV o", results_dir)
    print("[STAT] (neu chay session moi: nhin lai log [RESTORE] cell truoc day - phai co CSV duoc restore)")
else:
    frames = [pd.read_csv(f) for f in csv_files]
    combined = pd.concat(frames, ignore_index=True)
    print(f"[STAT] Gom {len(frames)} file -> {len(combined)} rows (round x seed x pipeline)")
    if "map50" not in combined.columns or "batch_diversity" not in combined.columns:
        print("[STAT][LOI] File CSV khong co cot map50/batch_diversity - kiem tra lai danh sach file")
    elif len(combined) % 2 != 0 or len(combined) > 100:
        print(f"[STAT][CANH BAO] So rows khong hop le ({len(combined)}) - co the da lan file thua")

    # Cot batch_diversity trong CSV la string cua dict Python -> parse lai
    def parse_diversity(d):
        if isinstance(d, dict):
            return sum(v for v in d.values() if isinstance(v, (int, float)))
        try:
            return sum(ast.literal_eval(d).values())
        except Exception:
            return 0.0

    combined["total_entropy"] = combined["batch_diversity"].apply(parse_diversity)
    combined = combined.sort_values(["pipeline", "seed", "round"]).reset_index(drop=True)

    print("\n" + "="*70)
    print("SUMMARY: mAP50 do tren FIXED eval set (2000 anh, seed 2024)")
    print("="*70)
    print(combined.pivot_table(index="pipeline", columns="round",
                                values="map50", aggfunc="mean").round(4))

    print("\n" + "="*70)
    print("[H1] Entropy so sanh (paired theo seed, theo round)")
    print("="*70)
    piv_b = combined[combined["pipeline"] == "baseline"].pivot_table(
        index="seed", columns="round", values="total_entropy")
    piv_s = combined[combined["pipeline"] == "stratified"].pivot_table(
        index="seed", columns="round", values="total_entropy")
    print("Baseline:\n", piv_b.round(4))
    print("Stratified:\n", piv_s.round(4))

    for r in sorted(set(piv_b.columns) & set(piv_s.columns)):
        b = piv_b[r].dropna().values
        s = piv_s[r].dropna().values
        n = min(len(b), len(s))
        if n > 1:
            t, p = stats.ttest_rel(s[:n], b[:n])
            print(f"\nRound {r} (n={n}): t={t:.4f}, p={p:.6f}, "
                  f"meanB={b[:n].mean():.4f}, meanS={s[:n].mean():.4f}, "
                  f"sig(a=0.05)={'YES' if p < 0.05 else 'NO'}")

    print("\n" + "="*70)
    print("[H2] Tuong quan entropy - mAP")
    print("="*70)
    if len(combined) > 2:
        pr, pp = stats.pearsonr(combined["total_entropy"], combined["map50"])
        sr, sp = stats.spearmanr(combined["total_entropy"], combined["map50"])
        print(f"Pearson   r={pr:.4f} (p={pp:.4f})")
        print(f"Spearman  r={sr:.4f} (p={sp:.4f})")
        print("\nPer-pipeline:")
        for pipe in ["baseline", "stratified"]:
            sub = combined[combined["pipeline"] == pipe]
            if len(sub) > 2:
                r2, p2 = stats.pearsonr(sub["total_entropy"], sub["map50"])
                print(f"  {pipe}: r={r2:.4f} (p={p2:.4f}, n={len(sub)})")


In [ ]:
# ============================================================
# BƯỚC 2: DOMAIN-SPECIFIC EVALUATION (mAP50 theo bối cảnh)
# Chạy SAU cell thống kê. Dùng _shared_eval_metadata.csv (do
# ActiveLearningLoop.run() dump, chung cho mọi pipeline/seed)
# + thư mục _shared_eval (ảnh+label đã copy sẵn) để tạo
# sub-eval set theo domain rồi val model round cuối của mỗi
# experiment trên từng domain.
# Kết quả: bảng mAP50 domain x pipeline (mean/std 3 seeds).
# ============================================================
import glob, os, shutil, yaml
import pandas as pd
import numpy as np
import torch
from data.loader import YOLO_CLASS_NAMES

SHARED_EVAL_DIR = "data/processed/_shared_eval"
EVAL_META_CSV = f"{results_dir}/_shared_eval_metadata.csv"
LAST_ROUND = EXPERIMENT_CONFIG.get("num_rounds", 2)   # run() lặp range(0, num_rounds+1) -> round cuối = num_rounds
DOMAIN_ROOT = "data/processed/_domain_eval"

DOMAINS = {
    "night":          lambda m: m["timeofday"] == "night",
    "adverse_weather":lambda m: m["weather"] in ("rainy", "snowy", "foggy"),
    "daytime_ctrl":   lambda m: m["timeofday"] == "daytime",  # gia tri THAT cua BDD100K (audit: daytime/night/dawn dusk)
}
# "rare_scenes": các scene HIẾM (tần suất < 30% của scene phổ biến nhất)
# -> định nghĩa từ dữ liệu thật, không hard-code tên scene (tránh lệch
#   giữa các version label BDD100K: "parking lot" vs "parking_lot"...).

def find_best_pt(exp_name):
    pats = [f"runs/detect/experiments/runs/{exp_name}/round_{LAST_ROUND}/weights/best.pt",
            f"experiments/runs/{exp_name}/round_{LAST_ROUND}/weights/best.pt"]
    for p in pats:
        if os.path.exists(p):
            return p
    return None

if not os.path.exists(EVAL_META_CSV):
    print(f"[DOMAIN-EVAL] Không có {EVAL_META_CSV} - bỏ qua (chạy lại experiment trước)")
else:
    meta = pd.read_csv(EVAL_META_CSV)
    # rare_scenes: threshold 30% số lượng scene phổ biến nhất
    vc = meta["scene"].value_counts()
    rare_scenes = set(vc[vc < 0.3 * vc.max()].index) - {"undefined", "unknown", "street"}
    DOMAINS["rare_scenes"] = lambda m: m["scene"] in rare_scenes
    print(f"[DOMAIN-EVAL] rare_scenes (tần suất < {0.3:.0%} top): {sorted(rare_scenes)}")

    # 1) Dựng sub-eval set theo domain (hardlink ảnh+label, không tốn RAM/disk)
    for dname, pred in DOMAINS.items():
        sel = meta[meta.apply(lambda r: pred(r), axis=1)]
        if sel.empty:
            print(f"[DOMAIN-EVAL] {dname}: 0 ảnh - bỏ qua"); continue
        ddir = os.path.join(DOMAIN_ROOT, dname)
        for sub in ("images", "labels"):
            os.makedirs(os.path.join(ddir, sub), exist_ok=True)
        n_link = 0
        for _, row in sel.iterrows():
            fname = os.path.basename(str(row["filename"]))
            stem = os.path.splitext(fname)[0]
            for src_dst in [(f"{SHARED_EVAL_DIR}/images/{fname}", f"{ddir}/images/{fname}"),
                            (f"{SHARED_EVAL_DIR}/labels/{stem}.txt", f"{ddir}/labels/{stem}.txt")]:
                s, d = src_dst
                if os.path.exists(s) and not os.path.exists(d):
                    try:
                        os.link(s, d); n_link += 1
                    except OSError:
                        shutil.copy2(s, d)
        n_lab = len(list(glob.glob(f"{ddir}/labels/*.txt")))
        ycfg = {"path": os.path.abspath(ddir), "train": "images", "val": "images",
                "names": list(YOLO_CLASS_NAMES.values()) if isinstance(YOLO_CLASS_NAMES, dict) else list(YOLO_CLASS_NAMES)}
        with open(f"{ddir}/domain.yaml", "w") as f:
            yaml.dump(ycfg, f)
        print(f"[DOMAIN-EVAL] {dname}: {len(sel)} ảnh metadata / {n_lab} có label -> {f'{ddir}/domain.yaml'}")

    # 2) Val model round cuối của mỗi experiment trên từng domain
    from ultralytics import YOLO
    rows = []
    for pipeline in PIPELINES:
        for seed in SEEDS:
            exp = f"{pipeline}_seed{seed}"
            wpt = find_best_pt(exp)
            if not wpt:
                print(f"[DOMAIN-EVAL] {exp}: không thấy best.pt round {LAST_ROUND} - bỏ qua"); continue
            model = YOLO(wpt)
            for dname in DOMAINS:
                dyaml = f"{DOMAIN_ROOT}/{dname}/domain.yaml"
                if not os.path.exists(dyaml):
                    continue
                try:
                    res = model.val(data=dyaml, device=0, plots=False, verbose=False)
                    rows.append({"experiment": exp, "pipeline": pipeline, "seed": seed,
                                 "domain": dname, "map50": float(res.box.map50)})
                except Exception as e:
                    print(f"[DOMAIN-EVAL] {exp}/{dname} fail: {e}")
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    dmap = pd.DataFrame(rows)
    print("\n" + "="*70)
    print(f"DOMAIN-SPECIFIC mAP50 (round {LAST_ROUND}) theo pipeline (mean/std của 3 seeds)")
    print("="*70)
    if len(dmap):
        piv = dmap.pivot_table(index="domain", columns="pipeline", values="map50", aggfunc="mean")
        sds = dmap.pivot_table(index="domain", columns="pipeline", values="map50", aggfunc="std")
        out = pd.DataFrame({
            p: [f"{piv.loc[d, p]:.4f} ±{sds.loc[d, p].iloc[0]:.4f}"
                if p in sds and pd.notna(sds.loc[d, p].iloc[0]) else f"{piv.loc[d, p]:.4f}"
                for d in piv.index]
            for p in piv.columns}, index=piv.index)
        print(out.to_string())
        delta = dmap[dmap.domain == "adverse_weather"].groupby("pipeline")["map50"].mean()
        if all(p in delta.index for p in PIPELINES):
            print(f"\nDelta adverse_weather (stratified - baseline) = {delta.get('stratified', np.nan) - delta.get('baseline', np.nan):+.4f} mAP50")
    else:
        print("(không có kết quả - kiểm tra đường dẫn best.pt)")


In [ ]:
print("\n" + "="*70)
print("MAIN EXPERIMENT COMPLETED!")
print("="*70)